# 05 — Pipeline OBD-II Real: Dataset Barreto

**Proyecto:** VERA — Mantenimiento Predictivo Vehicular  
**Dataset:** OBD-II DS3 — Barreto et al. (14 drivers, 14 cars, daily routes)  
**Fuente:** Kaggle · https://www.kaggle.com/datasets/cephasax/obdii-ds3  
**Objetivo:** Entrenar y evaluar modelos de clasificación binaria para detectar anomalías en el motor (`engine_condition`: 0=Normal, 1=Anomalía) usando señales OBD-II reales.

**Variable respuesta:** `engine_condition` — etiquetada por umbrales técnicos (Arena et al., 2022; SAE J1979).

**Prioridad de métricas:** Se prioriza **Recall** sobre Accuracy. Un falso negativo tiene costo mayor que un falso positivo.

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# Añadir src/ al path para importar módulos locales
NOTEBOOK_DIR = Path().resolve()
ML_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(ML_ROOT))

from src.preprocessing import limpiar_columna_numerica

# ── Rutas del proyecto
RAW_DIR       = ML_ROOT / "data" / "raw" / "barreto"
PROCESSED_DIR = ML_ROOT / "data" / "processed"
FIGURES_DIR   = ML_ROOT / "figures" / "barreto"
MODELS_DIR    = ML_ROOT / "models" / "barreto"

for d in [PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Rutas configuradas:")
print(f"  RAW      : {RAW_DIR}")
print(f"  PROCESSED: {PROCESSED_DIR}")
print(f"  FIGURES  : {FIGURES_DIR}")
print(f"  MODELS   : {MODELS_DIR}")


## Paso 1 — Carga del Dataset Barreto

In [ ]:
OBD_COLS = [
    "ENGINE_COOLANT_TEMP",
    "ENGINE_LOAD",
    "ENGINE_RPM",
    "INTAKE_MANIFOLD_PRESSURE",
    "MAF",
    "SPEED",
    "SHORT TERM FUEL TRIM BANK 1",
    "THROTTLE_POS",
    "TIMING_ADVANCE",
]

csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Archivos CSV encontrados: {len(csv_files)}")
for f in csv_files:
    print(f"  {f.name}")

frames = []
for csv_path in csv_files:
    df_tmp = pd.read_csv(csv_path, low_memory=False)
    df_tmp["session"] = csv_path.stem
    frames.append(df_tmp)

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nDataset cargado: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Sesiones únicas: {df_raw['session'].nunique()}")
df_raw.head(3)

In [ ]:
df_raw = df_raw.rename(columns={
    "SHORT TERM FUEL TRIM BANK 1": "SHORT_TERM_FUEL_TRIM_BANK_1"
})

OBD_COLS = [
    "ENGINE_COOLANT_TEMP",
    "ENGINE_LOAD",
    "ENGINE_RPM",
    "INTAKE_MANIFOLD_PRESSURE",
    "MAF",
    "SPEED",
    "SHORT_TERM_FUEL_TRIM_BANK_1",
    "THROTTLE_POS",
    "TIMING_ADVANCE",
    "session",
]

cols_disponibles = [c for c in OBD_COLS if c in df_raw.columns]
df = df_raw[cols_disponibles].copy()

numeric_cols = [c for c in cols_disponibles if c != "session"]
for col in numeric_cols:
    df[col] = limpiar_columna_numerica(df[col])

print(f"Columnas disponibles: {cols_disponibles}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nValores nulos:\n{df[numeric_cols].isnull().sum()}")

## Paso 2 — Análisis Exploratorio de Datos (EDA)

In [ ]:
print("Estadísticas descriptivas — Señales OBD-II")
print("=" * 70)
df[numeric_cols].describe().round(2)

In [ ]:
plot_cols = [c for c in numeric_cols if c != "session"][:9]
n_cols_plot = 3
n_rows_plot = int(np.ceil(len(plot_cols) / n_cols_plot))

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(15, 4 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].hist(df[col].dropna(), bins=50, color="#2196F3", alpha=0.75, edgecolor="white")
    axes[i].set_title(col.replace("_", " "), fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Valor", fontsize=10)
    axes[i].set_ylabel("Frecuencia", fontsize=10)
    axes[i].grid(axis="y", alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribución de Señales OBD-II — Dataset Barreto",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "histogramas_obd2.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {FIGURES_DIR / 'histogramas_obd2.png'}")

In [ ]:
corr = df[plot_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={"size": 9}
)
ax.set_title("Matriz de Correlación — Señales OBD-II (Barreto)",
             fontsize=12, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_matrix.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {FIGURES_DIR / 'correlation_matrix.png'}")

> **¿Por qué se eliminó el filtro IQR?**
>
> El filtro IQR no es apropiado para el Autoencoder no supervisado dado que el modelo aprende el comportamiento normal del motor y detecta desviaciones por error de reconstrucción elevado. Aplicar IQR previo eliminaría precisamente las señales anómalas que el modelo debe aprender a identificar, además de descartar variaciones válidas de conducción agresiva presentes en el dataset. Con factor=3.0 se estaban eliminando **46,885 de 60,439 registros (77.57%)**, reduciendo innecesariamente el conjunto de entrenamiento y perdiendo señales clave de comportamiento extremo del motor.

## Paso 3 — Preprocesamiento No Supervisado


In [ ]:
FEATURE_COLS = [
    "ENGINE_COOLANT_TEMP", "ENGINE_LOAD", "ENGINE_RPM",
    "INTAKE_MANIFOLD_PRESSURE", "MAF", "SPEED",
    "SHORT_TERM_FUEL_TRIM_BANK_1", "THROTTLE_POS", "TIMING_ADVANCE",
]
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
X_all = df[feature_cols].copy()

# Imputar nulos con mediana
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_all)
X_df_feat = pd.DataFrame(X_imputed, columns=feature_cols)

# Normalizar con MinMaxScaler (rango 0-1, mejor para autoencoders)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_df_feat)

# Split cronológico 80/20 — SIN shuffle para respetar orden temporal
n_total = len(X_scaled)
n_train = int(n_total * 0.80)

X_train = X_scaled[:n_train]
X_test  = X_scaled[n_train:]

print(f"Features usadas ({len(feature_cols)}): {feature_cols}")
print(f"\nTotal registros : {n_total:,}")
print(f"Train (80%)     : {n_train:,} registros")
print(f"Test  (20%)     : {n_total - n_train:,} registros")
print(f"\nRango MinMax — min: {X_scaled.min():.4f}, max: {X_scaled.max():.4f}")


## Paso 4 — Autoencoder (Detección No Supervisada)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

input_dim = len(feature_cols)

# ── Arquitectura: 9 → 16 → 8 → 4 → 8 → 16 → 9
inputs  = keras.Input(shape=(input_dim,), name="input")

# Encoder
x = layers.Dense(16, activation="relu", name="enc1")(inputs)
x = layers.Dense(8,  activation="relu", name="enc2")(x)
encoded = layers.Dense(4, activation="relu", name="bottleneck")(x)

# Decoder
x = layers.Dense(8,  activation="relu", name="dec1")(encoded)
x = layers.Dense(16, activation="relu", name="dec2")(x)
outputs = layers.Dense(input_dim, activation="linear", name="output")(x)

autoencoder = keras.Model(inputs, outputs, name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
)

print("Entrenando Autoencoder...")
history = autoencoder.fit(
    X_train, X_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

print(f"\nEntrenamiento completado en {len(history.history['loss'])} épocas")
print(f"  Train loss final : {history.history['loss'][-1]:.6f}")
print(f"  Val   loss final : {history.history['val_loss'][-1]:.6f}")


## Paso 5 — Detección de Anomalías

In [ ]:
# Reconstruction error en train → define threshold
X_train_pred = autoencoder.predict(X_train, verbose=0)
train_mse = np.mean(np.power(X_train - X_train_pred, 2), axis=1)

# Threshold = percentil 90 — garantiza ~10% de detección en conducción mixta
threshold = float(np.percentile(train_mse, 90))

# Reconstruction error en test
X_test_pred = autoencoder.predict(X_test, verbose=0)
test_mse = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

# Etiqueta: 1 = anomalía, 0 = normal
y_pred_ae = (test_mse > threshold).astype(int)
n_anomalias   = int(y_pred_ae.sum())
pct_anomalias = n_anomalias / len(y_pred_ae) * 100

print('=' * 55)
print('DETECCIÓN DE ANOMALÍAS — Autoencoder')
print('=' * 55)
print(f'  Threshold (P90 train MSE) : {threshold:.6f}')
print(f'  Total registros test      : {len(y_pred_ae):,}')
print(f'  Anomalías detectadas      : {n_anomalias:,}')
print(f'  % del test set            : {pct_anomalias:.2f}%')
print('=' * 55)


> **Justificación del umbral P90**
>
> El umbral de detección se establece en el percentil 90 del error de reconstrucción sobre el conjunto de entrenamiento. Esto implica que el 10% de las muestras con mayor error en train ya se consideran anómalas, un porcentaje razonable para datos de conducción mixta urbana/ruta donde se esperan variaciones normales de comportamiento. El percentil 95 (valor anterior) resultaba demasiado conservador para un dataset homogéneo sin fallas reales documentadas, produciendo 0 anomalías detectadas en test.

In [ ]:
# ── Análisis de anomalías por vehículo ──────────────────────────────────
# Debug: verificar alineación de índices
print(f'df_raw filas       : {len(df_raw):,}')
print(f'n_train            : {n_train:,}')
print(f'test_mse length    : {len(test_mse):,}')
print(f'VEHICLE_ID in cols : {"VEHICLE_ID" in df_raw.columns}')

# Recuperar VEHICLE_ID para el test set (últimas n_total - n_train filas)
if 'VEHICLE_ID' in df_raw.columns:
    vehicle_ids_test = df_raw['VEHICLE_ID'].iloc[n_train: n_train + len(test_mse)].reset_index(drop=True)
else:
    # Fallback: usar session como proxy de vehículo
    vehicle_ids_test = df['session'].iloc[n_train: n_train + len(test_mse)].reset_index(drop=True)
    print('AVISO: VEHICLE_ID no encontrado, usando session como proxy.')

results_df = pd.DataFrame({
    'VEHICLE_ID': vehicle_ids_test.values,
    'mse':        test_mse,
    'anomalia':   y_pred_ae,
})

vehicle_stats = (
    results_df.groupby('VEHICLE_ID')
    .agg(
        total_registros=('anomalia', 'count'),
        anomalias=('anomalia', 'sum'),
    )
    .assign(pct_anomalia=lambda d: (d['anomalias'] / d['total_registros'] * 100).round(2))
    .sort_values('pct_anomalia', ascending=False)
    .reset_index()
)

print(f'\nVehículos únicos en test: {len(vehicle_stats)}')
print(f'Total anomalías         : {n_anomalias:,} ({pct_anomalias:.2f}%)')
print('\n=== Anomalías por vehículo ===')
print(f'{"VEHICLE_ID":<15} {"total":>8} {"anomalias":>10} {"pct_%":>8}')
print('-' * 45)
for _, row in vehicle_stats.iterrows():
    print(f'{str(row["VEHICLE_ID"]):<15} {row["total_registros"]:>8,} {row["anomalias"]:>10,} {row["pct_anomalia"]:>7.2f}%')

if len(vehicle_stats) > 0:
    top_vehicle = vehicle_stats.iloc[0]
    print(f'\n→ Vehículo con mayor tasa: {top_vehicle["VEHICLE_ID"]} '
          f'({top_vehicle["pct_anomalia"]:.2f}% — '
          f'{int(top_vehicle["anomalias"]):,} / {int(top_vehicle["total_registros"]):,})')
else:
    top_vehicle = None
    print('No se detectaron anomalías — considera bajar el threshold.')


In [ ]:
# ── Feature más responsable por registro anómalo ──────────────────────────
X_test_arr      = X_test                        # shape (n_test, 9)
X_test_pred_arr = X_test_pred                   # shape (n_test, 9)

# Error por feature en cada registro del test
per_feature_sq_err = np.power(X_test_arr - X_test_pred_arr, 2)  # (n_test, 9)

# Para cada registro anómalo, identificar qué feature tuvo mayor error
anomaly_indices    = np.where(y_pred_ae == 1)[0]
dominant_features  = np.argmax(per_feature_sq_err[anomaly_indices], axis=1)
feature_counts     = np.bincount(dominant_features, minlength=len(feature_cols))

top3_resp_idx  = np.argsort(feature_counts)[::-1][:3]
top3_resp_feat = [(feature_cols[i], feature_counts[i]) for i in top3_resp_idx]

print('\n=== Top 3 features más frecuentemente responsables de anomalías ===')
print(f'{"Feature":<35} {"# veces dominante":>18} {"% de anomalías":>15}')
print('-' * 70)
for feat, cnt in top3_resp_feat:
    pct = cnt / len(anomaly_indices) * 100 if len(anomaly_indices) > 0 else 0
    print(f'{feat:<35} {cnt:>18,} {pct:>14.1f}%')


## Paso 6 — Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Histograma del reconstruction error
axes[0].hist(test_mse[y_pred_ae == 0], bins=80, color="#42A5F5",
             alpha=0.7, label="Normal", density=True)
axes[0].hist(test_mse[y_pred_ae == 1], bins=80, color="#EF5350",
             alpha=0.8, label="Anomalía", density=True)
axes[0].axvline(threshold, color="black", linestyle="--", lw=2,
                label=f"Threshold = {threshold:.5f}")
axes[0].set_title("Error de Reconstrucción por Clase", fontsize=12, fontweight="bold")
axes[0].set_xlabel("MSE por muestra", fontsize=10)
axes[0].set_ylabel("Densidad", fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

# ── Plot 2: Scatter temporal
idx = np.arange(len(test_mse))
axes[1].scatter(idx[y_pred_ae == 0], test_mse[y_pred_ae == 0],
                c="#42A5F5", s=2, alpha=0.3, label="Normal")
axes[1].scatter(idx[y_pred_ae == 1], test_mse[y_pred_ae == 1],
                c="#EF5350", s=8, alpha=0.9, label="Anomalía")
axes[1].axhline(threshold, color="black", linestyle="--", lw=1.5, label="Threshold")
axes[1].set_title("Error en el Tiempo — Test Set", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Índice temporal", fontsize=10)
axes[1].set_ylabel("MSE", fontsize=10)
axes[1].legend(fontsize=9, markerscale=4)
axes[1].grid(alpha=0.3)

# ── Plot 3: Top 3 features con mayor error en registros anómalos
anomaly_mask    = y_pred_ae == 1
X_test_df_orig  = pd.DataFrame(X_test, columns=feature_cols)
X_test_pred_df  = pd.DataFrame(X_test_pred, columns=feature_cols)

feature_errors_anomalies = np.mean(
    np.power(
        X_test_df_orig[anomaly_mask].values - X_test_pred_df[anomaly_mask].values, 2
    ),
    axis=0,
)
top3_idx      = np.argsort(feature_errors_anomalies)[::-1][:3]
top3_features = [feature_cols[i] for i in top3_idx]
top3_errors   = feature_errors_anomalies[top3_idx]

bar_colors = ["#EF5350", "#FF7043", "#FFA726"]
axes[2].barh(top3_features[::-1], top3_errors[::-1], color=bar_colors[::-1], edgecolor="white")
axes[2].set_title("Top 3 Features más Anómalas\n(MSE promedio en anomalías)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("MSE promedio de reconstrucción", fontsize=10)
axes[2].grid(axis="x", alpha=0.3)

plt.suptitle("Autoencoder — Detección de Anomalías OBD-II (Barreto)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "autoencoder_anomaly_detection.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"Figura guardada: {FIGURES_DIR / 'autoencoder_anomaly_detection.png'}")


## Paso 7 — Serialización del Modelo


In [ ]:
# Guardar artefactos del modelo
autoencoder.save(str(MODELS_DIR / 'barreto_autoencoder.keras'))
joblib.dump(scaler,       MODELS_DIR / 'barreto_scaler_minmax.pkl')
joblib.dump(imputer,      MODELS_DIR / 'barreto_imputer.pkl')
joblib.dump(feature_cols, MODELS_DIR / 'barreto_feature_names.pkl')
joblib.dump(threshold,    MODELS_DIR / 'barreto_ae_threshold.pkl')

print('Artefactos guardados:')
for f in sorted(MODELS_DIR.iterdir()):
    print(f'  {f.name:<45} {f.stat().st_size / 1024:.1f} KB')

print('\n' + '=' * 65)
print('RESUMEN FINAL — Autoencoder OBD-II (Barreto)')
print('=' * 65)
print(f'  Arquitectura         : {input_dim} → 16 → 8 → 4 → 8 → 16 → {input_dim}')
print(f'  Épocas entrenadas    : {len(history.history["loss"])}')
print(f'  Train loss final     : {history.history["loss"][-1]:.6f}')
print(f'  Threshold (P90 MSE)  : {threshold:.6f}')
print(f'  Anomalías detectadas : {n_anomalias:,} / {len(y_pred_ae):,} ({pct_anomalias:.2f}%)')
print(f'  Vehículo más anómalo : {top_vehicle["VEHICLE_ID"]} ({top_vehicle["pct_anomalia"]:.2f}%)')
print(f'\n  Top 3 features responsables de anomalías:')
for i, (feat, cnt) in enumerate(top3_resp_feat, 1):
    pct = cnt / len(anomaly_indices) * 100 if len(anomaly_indices) > 0 else 0
    print(f'    {i}. {feat:<38} {pct:.1f}% de los casos')
print('=' * 65)
